# Parsing de Documentos — Parte 3: PDFs Simples

**Problema:** documentos em PDF nativo (digital). Precisamos extrair texto e tabelas.

**Diferença fundamental:**
- **PDF nativo** (digital): gerado por Word/sistema — o texto existe como dados
- **PDF escaneado**: é uma imagem — só pixels, sem texto

**Ferramentas:**
- `PyMuPDF (fitz)` — extração rápida de texto corrido
- `pdfplumber` — extração de tabelas com estrutura preservada

In [ ]:
!pip install PyMuPDF pdfplumber reportlab -q

In [ ]:
import fitz  # PyMuPDF
import pdfplumber
import io
import re
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

---
## Bloco 3a — PyMuPDF: extração rápida de texto corrido

PyMuPDF é muito rápido para extrair texto de PDFs nativos.
Vamos criar um PDF de política interna e extrair o conteúdo.

In [ ]:
# Criar PDF de exemplo: política interna de home office
buffer = io.BytesIO()
c = canvas.Canvas(buffer, pagesize=A4)
largura, altura = A4

c.setFont("Helvetica-Bold", 16)
c.drawString(72, altura - 72, "POLÍTICA DE HOME OFFICE")

c.setFont("Helvetica", 10)
c.drawString(72, altura - 95, "Versão 2.1  |  Vigência: 01/01/2026  |  Classificação: Interno")

paragrafos = [
    ("1. OBJETIVO", True),
    ("Esta política estabelece as diretrizes para o regime de trabalho remoto (home office)", False),
    ("aplicável a todos os colaboradores da Empresa XYZ Ltda (CNPJ: 12.345.678/0001-99).", False),
    ("", False),
    ("2. ELEGIBILIDADE", True),
    ("2.1 São elegíveis colaboradores com mais de 6 meses de empresa.", False),
    ("2.2 O gestor direto deve aprovar a adesão ao regime remoto.", False),
    ("2.3 Estagiários não são elegíveis para home office integral.", False),
    ("", False),
    ("3. JORNADA DE TRABALHO", True),
    ("3.1 A jornada permanece de 8 horas diárias, das 09:00 às 18:00.", False),
    ("3.2 Intervalo obrigatório de 1 hora para almoço.", False),
    ("3.3 Horas extras devem ser previamente autorizadas pelo gestor.", False),
    ("", False),
    ("4. EQUIPAMENTOS", True),
    ("A empresa fornecerá: notebook corporativo, headset e auxílio mensal de R$ 150,00", False),
    ("para despesas de internet e energia elétrica.", False),
    ("", False),
    ("5. SEGURANÇA DA INFORMAÇÃO", True),
    ("5.1 É obrigatório o uso de VPN corporativa para acesso aos sistemas.", False),
    ("5.2 Documentos confidenciais não devem ser impressos em ambiente doméstico.", False),
    ("5.3 Incidentes de segurança devem ser reportados em até 24 horas.", False),
]

y = altura - 130
for texto, negrito in paragrafos:
    if negrito:
        c.setFont("Helvetica-Bold", 11)
    else:
        c.setFont("Helvetica", 10)
    c.drawString(72, y, texto)
    y -= 16

c.save()
pdf_texto_bytes = buffer.getvalue()
print(f"PDF criado em memória: {len(pdf_texto_bytes):,} bytes")

In [ ]:
# Extrair texto com PyMuPDF
doc = fitz.open(stream=pdf_texto_bytes, filetype="pdf")
pagina = doc[0]
texto_extraido = pagina.get_text()
doc.close()

print("=" * 60)
print("TEXTO EXTRAÍDO COM PyMuPDF")
print("=" * 60)
print(texto_extraido)

# Aplicar regex para encontrar padrões
print("\n" + "=" * 60)
print("PADRÕES ENCONTRADOS COM REGEX")
print("=" * 60)
cnpjs = re.findall(r'\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}', texto_extraido)
valores = re.findall(r'R\$\s?[\d.,]+', texto_extraido)
horarios = re.findall(r'\d{2}:\d{2}', texto_extraido)
print(f"CNPJs:     {cnpjs}")
print(f"Valores:   {valores}")
print(f"Horários:  {horarios}")

PyMuPDF extraiu o texto limpo e na ordem correta. Perfeito para texto corrido.

---
## Bloco 3b — PyMuPDF falha em tabelas

Agora vamos criar um PDF com tabela (folha de pagamento) e ver o que acontece
quando PyMuPDF tenta extrair.

In [ ]:
# Criar PDF com tabela: folha de pagamento
buffer_tab = io.BytesIO()
doc_tab = SimpleDocTemplate(buffer_tab, pagesize=A4)
styles = getSampleStyleSheet()

dados_tabela = [
    ["Funcionário",     "Cargo",             "Salário Base", "Desconto INSS", "Líquido"],
    ["Ana Paula Costa", "Analista Sênior",   "R$ 8.500,00",  "R$ 935,00",    "R$ 7.565,00"],
    ["Bruno Martins",   "Desenvolvedor Jr",  "R$ 4.200,00",  "R$ 462,00",    "R$ 3.738,00"],
    ["Carla Rodrigues", "Gestora de Produto","R$ 12.000,00", "R$ 1.320,00",  "R$ 10.680,00"],
    ["Diego Ferreira",  "Estagiário",        "R$ 1.800,00",  "R$ 0,00",      "R$ 1.800,00"],
    ["TOTAL",           "",                  "R$ 26.500,00", "R$ 2.717,00",  "R$ 23.783,00"],
]

tabela = Table(dados_tabela, colWidths=[120, 110, 85, 85, 85])
tabela.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2c3e50')),
    ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
    ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0, 0), (-1, -1), 9),
    ('GRID',       (0, 0), (-1, -1), 0.5, colors.grey),
    ('ALIGN',      (2, 0), (-1, -1), 'RIGHT'),
    ('BACKGROUND', (0, -1), (-1, -1), colors.HexColor('#bdc3c7')),
    ('FONTNAME',   (0, -1), (-1, -1), 'Helvetica-Bold'),
]))

titulo = Paragraph("<b>FOLHA DE PAGAMENTO — MARÇO/2026</b>", styles['Title'])
doc_tab.build([titulo, Spacer(1, 20), tabela])
pdf_tabela_bytes = buffer_tab.getvalue()
print(f"PDF com tabela criado: {len(pdf_tabela_bytes):,} bytes")

In [ ]:
# Tentar extrair a tabela com PyMuPDF
doc = fitz.open(stream=pdf_tabela_bytes, filetype="pdf")
texto_pymupdf = doc[0].get_text()
doc.close()

print("=" * 60)
print("PyMuPDF TENTANDO EXTRAIR A TABELA")
print("=" * 60)
print(texto_pymupdf)
print("\n⚠️  As colunas se misturaram! PyMuPDF lê texto na ordem do PDF,")
print("   mas não entende a estrutura de grade da tabela.")
print("   'Salário Base' pode ficar colado com 'Cargo', valores perdem alinhamento.")

---
## Bloco 3c — pdfplumber: extração de tabelas

pdfplumber analisa as linhas/bordas do PDF para reconstruir a grade da tabela.

In [ ]:
# Salvar PDF temporariamente para o pdfplumber (precisa de arquivo)
caminho_pdf = '/content/folha_pagamento.pdf'
with open(caminho_pdf, 'wb') as f:
    f.write(pdf_tabela_bytes)

# Extrair tabela com pdfplumber
with pdfplumber.open(caminho_pdf) as pdf:
    pagina = pdf.pages[0]
    tabela_extraida = pagina.extract_table()

print("=" * 60)
print("pdfplumber — TABELA EXTRAÍDA COM ESTRUTURA")
print("=" * 60)
if tabela_extraida:
    for i, linha in enumerate(tabela_extraida):
        if i == 0:
            print(' | '.join(f'{str(c):20s}' for c in linha))
            print('-' * 115)
        else:
            print(' | '.join(f'{str(c):20s}' for c in linha))
    print(f"\n✓ Estrutura preservada: {len(tabela_extraida)-1} linhas de dados")
else:
    print("Nenhuma tabela detectada.")

In [ ]:
# Comparação lado a lado
print("=" * 60)
print("COMPARAÇÃO: PyMuPDF vs pdfplumber")
print("=" * 60)

print("\n--- PyMuPDF (texto corrido, sem estrutura) ---")
print(texto_pymupdf[:400])

print("\n--- pdfplumber (tabela estruturada) ---")
if tabela_extraida:
    for linha in tabela_extraida[:3]:
        print(linha)
    print("...")

print("\n" + "=" * 60)
print("CONCLUSÃO:")
print("• PyMuPDF: rápido para texto corrido, perde estrutura de tabelas")
print("• pdfplumber: preserva tabelas, mas mais lento")
print("• Na prática, use os dois: PyMuPDF para texto, pdfplumber para tabelas")

---
## Trade-offs

| Ferramenta | ✅ Vantagens | ❌ Limitações |
|---|---|---|
| **PyMuPDF** | Muito rápido, texto limpo, leve | Perde estrutura de tabelas, não faz OCR |
| **pdfplumber** | Preserva tabelas, exporta para pandas | Mais lento, falha em tabelas sem bordas, não faz OCR |

> Na prática, PyMuPDF e pdfplumber **se complementam** para PDFs simples.
> Mas para documentos complexos (hierarquia, seções, mix texto+tabelas),
> eles perdem a **estrutura semântica** — um título vira texto igual a um parágrafo.
>
> **Próximo passo:** ferramentas que entendem a hierarquia do documento.